In [1]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [2]:
import numpy as np
import re
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from scipy.special import softmax

In [3]:
conflicts = train_df[train_df.duplicated(['text'], keep=False)].groupby('text')['target'].nunique()
train_df = train_df[~train_df['text'].isin(conflicts[conflicts > 1].index)].drop_duplicates(subset=['text']).copy()

def minimal_preprocess(text):
    text = re.sub(r'http\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    return re.sub(r'\s+', ' ', text).strip()

train_df['text'] = train_df['text'].apply(minimal_preprocess)
test_df['text'] = test_df['text'].apply(minimal_preprocess)

In [4]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

model zmieniony na roberta, nazwy zmiennych odnoszące się do poprzendich prób z deberta:

In [6]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=72)

tokenized_test = Dataset.from_pandas(test_df[['id', 'text']]).map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
test_probs_deberta = np.zeros((len(test_df), 2))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Konfiguracja
training_args = TrainingArguments(
    output_dir="./results_roberta",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=50,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none"
)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['target'])):
    print(f"\n--- Trenowanie DeBERTa Foldu {fold + 1}/3 ---")
    train_fold = train_df.iloc[train_idx][['text', 'target']].rename(columns={'target': 'label'})
    val_fold = train_df.iloc[val_idx][['text', 'target']].rename(columns={'target': 'label'})

    tokenized_train_fold = Dataset.from_pandas(train_fold).map(tokenize_function, batched=True)
    tokenized_val_fold = Dataset.from_pandas(val_fold).map(tokenize_function, batched=True)

    model_fold = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    trainer_fold = Trainer(
        model=model_fold,
        args=training_args,
        train_dataset=tokenized_train_fold,
        eval_dataset=tokenized_val_fold,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer_fold.train()
    preds_output = trainer_fold.predict(tokenized_test)
    test_probs_deberta += softmax(preds_output.predictions, axis=1) / skf.n_splits


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

... (16 outputs left out)

Map:   0%|          | 0/2495 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.450415,0.813226,0.788374
2,0.425964,0.456083,0.828858,0.795007
3,0.425964,0.499738,0.824850,0.790005


In [7]:
deberta_preds = np.argmax(test_probs_deberta, axis=1)
pd.DataFrame({'id': test_df['id'], 'target': deberta_preds}).to_csv('submission_roberta.csv', index=False)

In [8]:
# Wczytanie poprzedniego, najlepszego wyniku (BERTweet bez lokalizacji z wynikiem 0.84554)
bertweet_sub = pd.read_csv('submission_kfold.csv')
deberta_confidence = np.max(test_probs_deberta, axis=1)

final_ensemble_preds = []
for i in range(len(test_df)):
    pred_b = bertweet_sub.loc[i, 'target']
    pred_d = deberta_preds[i]

    if pred_b == pred_d:
        final_ensemble_preds.append(pred_d)
    else:
        if deberta_confidence[i] > 0.60:
            final_ensemble_preds.append(pred_d)
        else:
            final_ensemble_preds.append(pred_b)

ensemble_submission = pd.DataFrame({'id': test_df['id'], 'target': final_ensemble_preds})
ensemble_submission.to_csv('submission_ensemble_final.csv', index=False)